# 3. Comprehensive Model Comparison

Testing multiple models to improve prediction performance on log_shares.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import cross_validate, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, VotingRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Load preprocessed data
df = pd.read_csv('../data/preprocessed.csv')
print(f"Data shape: {df.shape}")
print(f"\nColumns: {list(df.columns)[:10]}...")
print(f"\nTarget (log_shares) stats:")
print(df['log_shares'].describe())

Data shape: (34896, 32)

Columns: ['n_tokens_title', 'n_tokens_content', 'num_hrefs', 'num_self_hrefs', 'average_token_length', 'num_keywords', 'data_channel_is_lifestyle', 'data_channel_is_entertainment', 'data_channel_is_bus', 'data_channel_is_socmed']...

Target (log_shares) stats:
count    34896.000000
mean         7.455805
std          0.916091
min          0.693147
25%          6.845880
50%          7.244942
75%          7.901377
max         13.645079
Name: log_shares, dtype: float64


In [3]:
# Prepare data
X = df.drop(columns=['log_shares'])
y = df['log_shares']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

Features shape: (34896, 31)
Target shape: (34896,)

Feature columns: ['n_tokens_title', 'n_tokens_content', 'num_hrefs', 'num_self_hrefs', 'average_token_length', 'num_keywords', 'data_channel_is_lifestyle', 'data_channel_is_entertainment', 'data_channel_is_bus', 'data_channel_is_socmed', 'data_channel_is_tech', 'data_channel_is_world', 'weekday_is_monday', 'weekday_is_tuesday', 'weekday_is_wednesday', 'weekday_is_thursday', 'weekday_is_friday', 'is_weekend', 'global_subjectivity', 'global_sentiment_polarity', 'global_rate_positive_words', 'avg_positive_polarity', 'avg_negative_polarity', 'title_subjectivity', 'title_sentiment_polarity', 'missing_kw_info', 'log_avg_shares_min_kw', 'log_avg_shares_max_kw', 'log_avg_shares_reference', 'num_imgs_cat', 'num_videos_cat']


In [4]:
def evaluate_model(model, X, y, cv=10, model_name="Model"):
    """
    Evaluate a model using cross-validation.
    Returns dict with mean and std metrics.
    """
    kf = KFold(n_splits=cv, shuffle=True, random_state=42)
    scaler = RobustScaler()  # Use RobustScaler for robustness to outliers
    
    mse_scores = []
    mae_scores = []
    r2_scores = []
    rmse_scores = []
    
    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        # Scale features
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Fit model
        model.fit(X_train_scaled, y_train)
        
        # Predict
        y_pred = model.predict(X_test_scaled)
        
        # Metrics in log space
        mse_scores.append(mean_squared_error(y_test, y_pred))
        mae_scores.append(mean_absolute_error(y_test, y_pred))
        rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))
        r2_scores.append(r2_score(y_test, y_pred))
    
    return {
        'name': model_name,
        'r2_mean': np.mean(r2_scores),
        'r2_std': np.std(r2_scores),
        'mae_mean': np.mean(mae_scores),
        'mae_std': np.std(mae_scores),
        'rmse_mean': np.mean(rmse_scores),
        'rmse_std': np.std(rmse_scores),
        'mse_mean': np.mean(mse_scores),
    }

print("Evaluation function defined")

Evaluation function defined


In [ ]:
# Test different models
models_to_test = [
    (LinearRegression(), "Linear Regression (Baseline)"),
    (Ridge(alpha=1.0), "Ridge Regression"),
    (Lasso(alpha=0.1, max_iter=10000), "Lasso Regression"),
    (ElasticNet(alpha=0.1, max_iter=10000), "ElasticNet"),
    (RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), "Random Forest (100 trees)"),
    (RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1), "Random Forest (200 trees)"),
    (GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42), "Gradient Boosting"),
    (GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42), "GB (200 trees, lr=0.05)"),
    (AdaBoostRegressor(n_estimators=100, random_state=42), "AdaBoost"),
    (SVR(kernel='rbf', C=100, gamma='scale'), "SVR (RBF kernel)"),
    (MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42), "Neural Network (100-50)"),
]

results = []
print("\n" + "="*80)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*80 + "\n")

for model, model_name in models_to_test:
    print(f"Testing {model_name}...", end=" ", flush=True)
    result = evaluate_model(model, X, y, cv=10, model_name=model_name)
    results.append(result)
    print(f"✓ (R²: {result['r2_mean']:.4f})")

print("\nAll models tested!")


COMPREHENSIVE MODEL COMPARISON

Testing Linear Regression (Baseline)... ✓ (R²: 0.1084)
Testing Ridge Regression... ✓ (R²: 0.1084)
Testing Lasso Regression... ✓ (R²: 0.0287)
Testing ElasticNet... ✓ (R²: 0.0496)
Testing Random Forest (100 trees)... ✓ (R²: 0.1387)
Testing Random Forest (200 trees)... ✓ (R²: 0.1444)
Testing Gradient Boosting... ✓ (R²: 0.1555)
Testing GB (200 trees, lr=0.05)... ✓ (R²: 0.1551)
Testing AdaBoost... ✓ (R²: -0.3574)
Testing SVR (RBF kernel)... 

In [ ]:
# Display results in a table
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('r2_mean', ascending=False)

print("\n" + "="*100)
print("MODEL COMPARISON RESULTS (Sorted by R²)")
print("="*100 + "\n")

display_cols = ['name', 'r2_mean', 'r2_std', 'mae_mean', 'rmse_mean']
for idx, row in results_df.iterrows():
    print(f"\n{idx+1}. {row['name']}")
    print(f"   R²:    {row['r2_mean']:>8.4f} ± {row['r2_std']:.4f}")
    print(f"   MAE:   {row['mae_mean']:>8.2f}")
    print(f"   RMSE:  {row['rmse_mean']:>8.2f}")

print("\n" + "="*100)

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# R² Score
ax = axes[0]
colors = ['green' if x > 0 else 'red' for x in results_df['r2_mean']]
ax.barh(range(len(results_df)), results_df['r2_mean'], color=colors, alpha=0.7)
ax.set_yticks(range(len(results_df)))
ax.set_yticklabels(results_df['name'], fontsize=9)
ax.set_xlabel('R² Score', fontsize=11)
ax.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
ax.set_title('R² Score Comparison', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# MAE
ax = axes[1]
ax.barh(range(len(results_df)), results_df['mae_mean'], color='steelblue', alpha=0.7)
ax.set_yticks(range(len(results_df)))
ax.set_yticklabels(results_df['name'], fontsize=9)
ax.set_xlabel('Mean Absolute Error', fontsize=11)
ax.set_title('MAE Comparison', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# RMSE
ax = axes[2]
ax.barh(range(len(results_df)), results_df['rmse_mean'], color='coral', alpha=0.7)
ax.set_yticks(range(len(results_df)))
ax.set_yticklabels(results_df['name'], fontsize=9)
ax.set_xlabel('Root Mean Squared Error', fontsize=11)
ax.set_title('RMSE Comparison', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved to ../figures/model_comparison.png")

In [ ]:
# Recommend best model
best_model = results_df.iloc[0]
print("\n" + "="*80)
print("RECOMMENDATION")
print("="*80)
print(f"\n✓ Best Model: {best_model['name']}")
print(f"  - R² Score: {best_model['r2_mean']:.4f}")
print(f"  - MAE: {best_model['mae_mean']:.2f}")
print(f"  - RMSE: {best_model['rmse_mean']:.2f}")
print(f"\nImprovement over baseline (R² = -0.0084):")
improvement = best_model['r2_mean'] - (-0.0084)
print(f"  + {improvement:.4f} ({improvement/abs(-0.0084)*100:.1f}%)")
print("\n" + "="*80)

In [ ]:
# Export results to CSV
results_df.to_csv('../data/model_comparison_results.csv', index=False)
print("Results saved to ../data/model_comparison_results.csv")